In [1]:
import random, spacy
from datasets import load_dataset, Dataset, DatasetDict
from nltk.corpus import wordnet

In [2]:
ds = load_dataset('glue', 'mrpc')

In [3]:
nlp = spacy.load("en_core_web_sm")
' '.join(token.pos_ for token in nlp("The quick brown fox jumped over the lazy dog."))

'DET ADJ ADJ NOUN VERB ADP DET ADJ NOUN PUNCT'

In [ ]:
def make_prompt(row):
    s1 = row['sentence1']
    s2 = row['sentence2']
    label = "Yes" if row['label'] else "No"
    return f"Sentence1: {s1}\nSentence2: {s2}\nDo these sentences mean the same thing? Write 'Yes' if they do, or 'No' if they don't.\n{label}"

In [5]:
def augment_sentence(sentence):
    nouns = [token.text for token in nlp(sentence) if token.pos_ == 'ADJ']
    if len(nouns) == 0: return sentence

    for noun in nouns:
        synonyms = wordnet.synsets(noun, pos='a') # pos=n retrieves only nouns
        synonyms = [s.name().replace('_',' ')[:-5] for s in synonyms] # last 5 characters are not part of the word (ex. table.n.01)
        synonyms = [s for s in synonyms if s != noun]
        if len(synonyms) == 0: continue
        sentence = sentence.replace(noun, random.choice(synonyms))

    return sentence

augment_sentence("The quick brown fox jumped over the lazy dog.")

'The agile brown fox jumped over the faineant dog.'

In [ ]:
train_augmented = {'text': []}

for row in ds['train']:
    prompt = make_prompt(row)
    
    train_augmented['text'].append(prompt)

    row['sentence1'] = augment_sentence(row['sentence1'])
    row['sentence2'] = augment_sentence(row['sentence2'])
    prompt_augmented = make_prompt(row)
    if prompt_augmented != prompt:
        train_augmented['text'].append(prompt_augmented)

In [7]:
train = {
    'text': [make_prompt(row['sentence1'], row['sentence2'], "Yes" if row['label'] else "No") for row in ds['train']]
}

validation = {
    'text': [make_prompt(row['sentence1'], row['sentence2'], "") for row in ds['validation']],
    'label': ["Yes" if row['label'] else "No" for row in ds['validation']]
}

test = {
    'text': [make_prompt(row['sentence1'], row['sentence2'], "") for row in ds['test']],
    'label': ["Yes" if row['label'] else "No" for row in ds['test']]
}

In [9]:
DatasetDict({
    'train': Dataset.from_dict(train),
    'train-augmented': Dataset.from_dict(train_augmented),
    'validation': Dataset.from_dict(validation),
    'test': Dataset.from_dict(test)
}).save_to_disk('dataset.hf')

Saving the dataset (0/1 shards):   0%|          | 0/3668 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7336 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/408 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1725 [00:00<?, ? examples/s]